# Ejercicios Prácticos — Ingeniería de Datos con Pandas
### Basados en la Clase 4: Ingeniería de Datos con Pandas — Tema: E-commerce

Estos ejercicios usan **3 archivos** (`pedidos_online.csv`, `catalogo_productos_ecommerce.csv`, `clientes_ecommerce.json`) que simulan datos reales de una tienda online, con **problemas de calidad insertados intencionalmente** (nulos, duplicados, fechas malformadas) para que puedas practicar exactamente lo que viste en el notebook original: carga de datos, EDA, limpieza, `.dt`, `merge`, `groupby`/`agg`, `concat` y una consulta SQL con `sqlite3`.

Todo el contenido es generado y verificable por ti mismo ejecutando el código — no depende de fuentes externas, así que puedes comprobar cada resultado con `.shape`, `.isna().sum()`, `.dtypes`, etc.

> 📎 Antes de ejecutar, asegúrate de tener en la misma carpeta que este notebook: `pedidos_online.csv`, `catalogo_productos_ecommerce.csv`, `clientes_ecommerce.json`

## Estructura de los datos

**`pedidos_online.csv`** → `id_pedido, id_producto, cantidad, fecha, id_cliente, metodo_pago`
- 165 filas (160 originales + 5 duplicados exactos insertados, simulando doble clic en "Comprar")
- 21 valores nulos en `cantidad`
- Algunas fechas vienen con formato `AAAA/MM/DD` en lugar de `AAAA-MM-DD` (fechas inválidas para `pd.to_datetime`)

**`catalogo_productos_ecommerce.csv`** → `id_producto, producto, categoria, precio`
- 9 filas (8 productos + 1 duplicado exacto, simulando un error de sincronización del inventario)
- 1 valor nulo en `precio` (el de "Lámpara LED Escritorio")

**`clientes_ecommerce.json`** → `id_cliente, nombre, ciudad, cliente_premium`
- 20 clientes
- 1 valor nulo en `ciudad`

In [1]:
import pandas as pd
import numpy as np
import sqlite3

pd.set_option('display.max_columns', None)

---
## 🧩 Ejercicio 1 — Ingesta, EDA y Limpieza de Datos

**Objetivo:** practicar `read_csv`, `read_json`, `.info()`, `.describe()`, `isna()`, `fillna()`, `dropna()`, `duplicated()` y `drop_duplicates()`.

**Instrucciones:**

1. Carga los tres archivos (`pedidos_online.csv` con `parse_dates=['fecha']`, `catalogo_productos_ecommerce.csv`, `clientes_ecommerce.json` con `orient='records'`).
2. Haz una auditoría inicial de cada DataFrame con `.info()` y `.describe()`.
3. Cuenta los nulos por columna en cada tabla con `.isna().sum()`.
4. Trata los nulos con un criterio justificado (igual que en el notebook):
   - `cantidad` nulo en pedidos → imputa con `1` (mínimo lógico, se asume compra unitaria) y conviértelo a `int`.
   - `precio` nulo en catálogo → imputa con el precio promedio de su `categoria`.
   - `ciudad` nula en clientes → márcala como `'Ciudad Desconocida'`.
5. Detecta y elimina los duplicados:
   - Fila completamente duplicada en `catalogo_productos_ecommerce` (usa `duplicated(keep=False)` primero para verla, luego `drop_duplicates()`).
   - Duplicados exactos en `pedidos_online` (¿cuántas filas había antes y después?).
6. Convierte `fecha` con `pd.to_datetime(..., errors='coerce')`, identifica cuántas fechas quedaron como `NaT` y elimínalas con `dropna(subset=['fecha'])`.

**Pregúntate al final:** ¿cuántas filas tenías al inicio en `pedidos_online.csv` y cuántas te quedaron después de limpiar? Documenta cada decisión con un comentario, como pide la sección "Saber Ser" del notebook.

In [3]:
# 1. Carga de los archivos
df_pedidos = pd.read_csv('pedidos_online.csv', parse_dates=['fecha'])
df_productos = pd.read_csv('catalogo_productos_ecommerce.csv') 
df_clientes = pd.read_json('clientes_ecommerce.json', orient='records') 

print("\nArchivos Cargados Correctamente")



Archivos Cargados Correctamente


In [8]:
# 2. Auditoría inicial (.info() y .describe())
df_pedidos.info()
print()
df_productos.info()
print()
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165 entries, 0 to 164
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_pedido    165 non-null    int64  
 1   id_producto  165 non-null    object 
 2   cantidad     144 non-null    float64
 3   fecha        165 non-null    object 
 4   id_cliente   165 non-null    object 
 5   metodo_pago  165 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 7.9+ KB

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id_producto  9 non-null      object 
 1   producto     9 non-null      object 
 2   categoria    9 non-null      object 
 3   precio       8 non-null      float64
dtypes: float64(1), object(3)
memory usage: 420.0+ bytes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (tot

In [10]:
df_pedidos.describe()

,id_pedido,cantidad
count,165.00000,144.000000
mean,79.69697,2.111111
std,46.66662,1.078143
min,1.00000,1.000000
25%,40.00000,1.000000
50%,79.00000,2.000000
75%,120.00000,3.000000
max,160.00000,4.000000


In [11]:
df_clientes.describe()

,id_cliente,nombre,ciudad,cliente_premium
count,20,20,19,20
unique,20,20,5,2
top,CL01,Ana Torres,Bogotá,False
freq,1,1,7,13


In [12]:
df_productos.describe()

,precio
count,8.000000
mean,123000.000000
std,73546.875237
min,45000.000000
25%,62500.000000
50%,104500.000000
75%,180000.000000
max,250000.000000


In [13]:
# 3. Conteo de nulos por columna en cada tabla
df_clientes.isna().sum()

id_cliente         0
nombre             0
ciudad             1
cliente_premium    0
dtype: int64

In [14]:
df_pedidos.isna().sum()

id_pedido       0
id_producto     0
cantidad       21
fecha           0
id_cliente      0
metodo_pago     0
dtype: int64

In [15]:
df_productos.isna().sum()

id_producto    0
producto       0
categoria      0
precio         1
dtype: int64

In [17]:
# 4. Tratamiento de nulos
df_pedidos["cantidad"] = df_pedidos["cantidad"].fillna(1).astype(int)
df_productos["precio"] = df_productos["precio"].fillna(df_productos["categoria"].mean)
df_clientes["ciudad"] = df_clientes["ciudad"].fillna("Ciudad Desconocida")

In [23]:
# 5. Detección y eliminación de duplicados
df_clientes[df_clientes.duplicated()]

,id_cliente,nombre,ciudad,cliente_premium


In [25]:
df_productos[df_productos.duplicated(keep=False)]

,id_producto,producto,categoria,precio
4,SKU05,Teclado Mecánico,Electrónica,180000.0
8,SKU05,Teclado Mecánico,Electrónica,180000.0


In [22]:
df_pedidos[df_pedidos.duplicated()]

,id_pedido,id_producto,cantidad,fecha,id_cliente,metodo_pago
160,71,SKU06,3,2025-02-14,CL01,Contra entrega
161,140,SKU07,1,2025-03-09,CL15,PayPal
162,7,SKU08,3,2025-04-13,CL06,Tarjeta de débito
163,4,SKU04,4,2025-04-15,CL02,Tarjeta de crédito
164,48,SKU07,1,2025-02-12,CL15,Tarjeta de débito


In [ ]:
# 6. Conversión y limpieza de fechas


---
## 🧩 Ejercicio 2 — Ingeniería de Fechas + GroupBy/Agg

**Objetivo:** practicar el acceso `.dt`, `groupby()` con `.agg()`, y `merge` (inner/left).

Usa el DataFrame de pedidos ya limpio del Ejercicio 1.

**Instrucciones:**

1. Extrae de la columna `fecha`: `anio`, `mes`, `nombre_mes`, `dia_semana` y `trimestre` (usa `.dt`).
2. Une `pedidos_online` con `catalogo_productos_ecommerce` (`merge`, `how='inner'`, llave `id_producto`) y luego con `clientes_ecommerce` (`how='left'`, llave `id_cliente`).
3. Crea la columna `total_pedido = cantidad * precio`.
4. Responde con `groupby` + `.agg()`:
   - Ingresos totales, número de pedidos y ticket promedio **por ciudad**.
   - Unidades vendidas e ingresos **por categoría y producto**, ordenado de mayor a menor ingreso.
   - Comparación de ingresos y ticket promedio entre `cliente_premium` (`True` vs `False`).
   - ¿Cuál es el `metodo_pago` más usado? ¿Y cuál genera más ingresos en promedio? (`groupby('metodo_pago')`)
   - ¿Qué día de la semana (`dia_semana`) recibe más pedidos? (`value_counts()`)

In [ ]:
# 1. Ingeniería de fechas con .dt


In [ ]:
# 2. Merge: pedidos + catálogo (inner) + clientes (left)


In [ ]:
# 3. Columna total_pedido = cantidad * precio


In [ ]:
# 4. Ingresos, número de pedidos y ticket promedio por ciudad


In [ ]:
# 4. Unidades e ingresos por categoría y producto (ordenado desc)


In [ ]:
# 4. Comparación cliente_premium (True vs False)


In [ ]:
# 4. Método de pago más usado y el que más ingresos genera en promedio


In [ ]:
# 4. Día de la semana con más pedidos


---
## 🧩 Ejercicio 3 — Concat, Auditoría con Outer Join y SQL desde Pandas

**Objetivo:** practicar `pd.concat()`, `merge(how='outer', indicator=True)` y `pd.read_sql()` con `sqlite3`.

**Instrucciones:**

1. Divide el DataFrame de pedidos limpio en dos mitades simulando dos centros de distribución (`Bodega Norte` y `Bodega Sur`, igual que en el notebook) y únelas de nuevo con `pd.concat(axis=0, ignore_index=True)`. Verifica que el total de filas cuadre.
2. Haz un `merge(how='outer', indicator=True)` entre `pedidos_online` y `catalogo_productos_ecommerce` por `id_producto`. Usa `value_counts()` sobre la columna `_merge` para auditar: ¿hay productos en el catálogo que nunca se vendieron? ¿hay pedidos con un `id_producto` que no existe en el catálogo?
3. Crea una conexión SQLite en memoria (`sqlite3.connect(':memory:')`), carga el DataFrame completo (unido con clientes) como tabla `pedidos_completo` usando `.to_sql()`, y con `pd.read_sql()` responde estas 2 consultas en SQL puro:
   - **Top 3 productos por ingresos totales** (`GROUP BY`, `SUM`, `ORDER BY`, `LIMIT`).
   - **Ingresos por mes, solo los meses con más de 10 pedidos** (`strftime('%Y-%m', fecha)`, `GROUP BY`, `HAVING`).
4. Cierra la conexión con `conn.close()`.

In [ ]:
# 1. Split en dos "bodegas" y concat de nuevo


In [ ]:
# 2. Merge outer con indicator=True para auditoría


In [ ]:
# 3. Conexión SQLite en memoria + carga de la tabla
conn = sqlite3.connect(':memory:')
# TODO: df_completo.to_sql('pedidos_completo', conn, index=False, if_exists='replace')


In [ ]:
# 3a. Top 3 productos por ingresos totales (SQL)
query_top_productos = """
-- TODO: escribe tu consulta SQL aquí
"""
# pd.read_sql(query_top_productos, conn)


In [ ]:
# 3b. Ingresos por mes, solo meses con más de 10 pedidos (SQL)
query_ingresos_mes = """
-- TODO: escribe tu consulta SQL aquí
"""
# pd.read_sql(query_ingresos_mes, conn)


In [ ]:
# 4. Cerrar la conexión
conn.close()


---
## ✅ Criterios de verificación (para que revises tu propio trabajo)

- El `.shape` final de `pedidos_online` limpio debe ser menor al original (por duplicados y fechas inválidas eliminadas).
- `catalogo_productos_ecommerce` limpio debe tener 8 filas (una menos que el original, por el duplicado).
- Ninguna columna clave (`cantidad`, `precio`, `ciudad`) debe tener nulos después de la limpieza — confírmalo con `.isna().sum()`.
- La suma de `total_pedido` calculada con pandas debe coincidir con la suma de `ingresos` que te da la consulta SQL del Ejercicio 3 (son la misma información, calculada por dos caminos distintos — es una buena forma de verificar que no cometiste errores).

In [ ]:
# Espacio libre para tu verificación final
